In [ ]:

import json
import requests
import folium
from branca.element import Element

# ==================================================
# 1. BANK DATA
# ==================================================

# These country footprints and categories are illustrative.
# Verify them against the banks' annual reports before
# presenting them as factual findings.

banks = [
    {
        "bank": "BBVA", "lat": 40.4168, "lon": -3.7038,
        "hq_country": "ESP", "assets_bn": None,
        "category": "International",
        "countries": ["ESP", "TUR"]
    },
    {
        "bank": "Nordea", "lat": 60.1699, "lon": 24.9384,
        "hq_country": "FIN", "assets_bn": None,
        "category": "Regional",
        "countries": ["FIN", "SWE", "DNK", "NOR"]
    },
    {
        "bank": "Commerzbank", "lat": 50.1109, "lon": 8.6821,
        "hq_country": "DEU", "assets_bn": None,
        "category": "Home-market-focused",
        "countries": ["DEU", "POL"]
    },
    {
        "bank": "ING", "lat": 52.3676, "lon": 4.9041,
        "hq_country": "NLD", "assets_bn": None,
        "category": "International",
        "countries": ["NLD", "BEL", "DEU", "POL", "ROU"]
    },
    {
        "bank": "Intesa Sanpaolo", "lat": 45.4642, "lon": 9.1900,
        "hq_country": "ITA", "assets_bn": None,
        "category": "Regional",
        "countries": ["ITA", "SVN", "SVK", "HRV", "HUN"]
    },
    {
        "bank": "Millennium bcp", "lat": 38.7223, "lon": -9.1393,
        "hq_country": "PRT", "assets_bn": None,
        "category": "Regional",
        "countries": ["PRT", "POL"]
    },
    {
        "bank": "Jyske Bank", "lat": 56.1629, "lon": 9.5450,
        "hq_country": "DNK", "assets_bn": None,
        "category": "Home-market-focused",
        "countries": ["DNK"]
    },
    {
        "bank": "Eurobank", "lat": 37.9838, "lon": 23.7275,
        "hq_country": "GRC", "assets_bn": None,
        "category": "Regional",
        "countries": ["GRC", "CYP", "BGR", "LUX"]
    },
    {
        "bank": "Lloyds Banking Group",
        "lat": 51.5072, "lon": -0.1276,
        "hq_country": "GBR", "assets_bn": None,
        "category": "Home-market-focused",
        "countries": ["GBR"]
    },
    {
        "bank": "OTP Bank", "lat": 47.4979, "lon": 19.0402,
        "hq_country": "HUN", "assets_bn": None,
        "category": "International",
        "countries": [
            "HUN", "BGR", "HRV", "SRB",
            "SVN", "SVK", "ROU", "UKR"
        ]
    },
]

bank_colors = {
    "BBVA": "#377EB8",
    "Nordea": "#E41A1C",
    "Commerzbank": "#4DAF4A",
    "ING": "#984EA3",
    "Intesa Sanpaolo": "#FF7F00",
    "Millennium bcp": "#A65628",
    "Jyske Bank": "#F781BF",
    "Eurobank": "#17BECF",
    "Lloyds Banking Group": "#BCBD22",
    "OTP Bank": "#636363",
}

# ==================================================
# 2. DOWNLOAD COUNTRY BOUNDARIES
# ==================================================

geojson_url = (
    "https://raw.githubusercontent.com/nvkelso/"
    "natural-earth-vector/master/geojson/"
    "ne_110m_admin_0_countries.geojson"
)

response = requests.get(geojson_url, timeout=30)
response.raise_for_status()
world_geojson = response.json()

europe_features = [
    feature
    for feature in world_geojson["features"]
    if (
        feature.get("properties", {}).get("CONTINENT") == "Europe"
        or feature.get("properties", {}).get("ADM0_A3") in ["TUR", "CYP"]
    )
]

europe_geojson = {
    "type": "FeatureCollection",
    "features": europe_features,
}

country_features = {
    feature.get("properties", {}).get("ADM0_A3"): feature
    for feature in europe_features
}

# ==================================================
# 3. CREATE MAP WITH OPENSTREETMAP
# ==================================================

m = folium.Map(
    location=[50, 12],
    zoom_start=4,
    tiles="OpenStreetMap",
    control_scale=True,
)

# Draw neutral country borders first.
folium.GeoJson(
    europe_geojson,
    name="Country borders",
    style_function=lambda feature: {
        "fillColor": "#FFFFFF",
        "color": "#8A929B",
        "weight": 0.8,
        "fillOpacity": 0.08,
    },
    control=False,
).add_to(m)

# ==================================================
# 4. CREATE ONE LAYER PER BANK
# ==================================================

# Store actual Folium layer objects for later use.
bank_layers = {}

for bank in banks:
    name = bank["bank"]
    category = bank["category"]
    colour = bank_colors[name]

    bank_layer = folium.FeatureGroup(
        name=name,
        show=True,
        control=False,
    )

    # Country footprint for this bank.
    footprint_features = [
        country_features[code]
        for code in bank["countries"]
        if code in country_features
    ]

    if footprint_features:
        footprint_geojson = {
            "type": "FeatureCollection",
            "features": footprint_features,
        }

        folium.GeoJson(
            footprint_geojson,
            style_function=lambda feature, c=colour: {
                "fillColor": c,
                "color": c,
                "weight": 1.2,
                "fillOpacity": 0.32,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["ADMIN"],
                aliases=["Country:"],
                sticky=True,
            ),
        ).add_to(bank_layer)

    # Headquarters marker.
    folium.CircleMarker(
        location=[bank["lat"], bank["lon"]],
        radius=7,
        color="#FFFFFF",
        weight=1.5,
        fill=True,
        fill_color=colour,
        fill_opacity=1,
        tooltip=f"{name} — {category}",
        popup=folium.Popup(
            f"<b>{name}</b><br>"
            f"Category: {category}<br>"
            f"Headquarters: {bank['hq_country']}",
            max_width=250,
        ),
    ).add_to(bank_layer)

    bank_layer.add_to(m)

    # IMPORTANT: retain the actual Python layer object.
    bank_layers[name] = bank_layer

# ==================================================
# 5. BUILD THE DROPDOWN FILTER PANEL
# ==================================================
categories = ["International", "Regional", "Home-market-focused"]


category_counts = {
    category: sum(
        bank["category"] == category for bank in banks
    )
    for category in categories
}

bank_legend_html = ""

for bank in banks:
    bank_legend_html += f"""
    <div class="legend-row">
        <span class="swatch"
              style="background:{bank_colors[bank['bank']]}"></span>
        {bank['bank']}
    </div>
    """
    
    
category_html = ""
for category in categories:
    category_html += f"""
    <label class="filter-row">
        <input type="checkbox"
               class="category-filter"
               data-category="{category}"
               checked>
        <span>{category}</span>
        <span class="count">{category_counts[category]}</span>
    </label>
    """
    

bank_html = ""

for bank in banks:
    bank_html += f"""
    <label class="filter-row">
        <input type="checkbox"
               class="bank-filter"
               data-bank="{bank['bank']}"
               data-category="{bank['category']}"
               checked>
        <span class="swatch"
              style="background:{bank_colors[bank['bank']]}"></span>
        <span>{bank['bank']}</span>
    </label>
    """

panel_html = f"""
<div id="map-control-panel">
  <details id="map-controls">
    <summary>Map controls</summary>

    <div class="panel-content">
      <div class="panel-heading">Filter by category</div>
      {category_html}

      <div class="panel-heading separator">
        Filter individual banks
      </div>

      <div class="bank-list">
        {bank_html}
      </div>

      <div class="button-row">
        <button type="button" id="select-all">Select all</button>
        <button type="button" id="clear-all">Clear all</button>
      </div>

      <div class="panel-heading separator">Legend</div>

      <div class="legend-row">
        <span class="swatch" style="background:#2878B5"></span>
        International
      </div>
      <div class="legend-row">
        <span class="swatch" style="background:#E69F00"></span>
        Regional
      </div>
      <div class="legend-row">
        <span class="swatch" style="background:#26966F"></span>
        Home-market-focused
      </div>
    </div>
  </details>
</div>
"""

css = """
<style>
#map-control-panel {
    position: fixed;
    top: 12px;
    left: 55px;
    z-index: 9999;
    font-family: Arial, sans-serif;
    font-size: 13px;
    color: #202733;
}
#map-controls {
    width: 285px;
    max-width: calc(100vw - 80px);
    background: rgba(255,255,255,0.98);
    border: 1px solid #CBD1D8;
    border-radius: 8px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.16);
}
#map-controls summary {
    cursor: pointer;
    font-weight: 700;
    padding: 11px 13px;
    list-style: none;
}
#map-controls summary::-webkit-details-marker {
    display: none;
}
#map-controls summary:after {
    content: " +";
    float: right;
}
#map-controls[open] summary:after {
    content: " −";
}
.panel-content {
    border-top: 1px solid #E2E6EA;
    padding: 10px 12px 12px;
}
.panel-heading {
    font-weight: 700;
    margin: 2px 0 8px;
}
.separator {
    border-top: 1px solid #E2E6EA;
    padding-top: 10px;
    margin-top: 10px;
}
.filter-row {
    display: flex;
    align-items: center;
    gap: 7px;
    margin: 7px 0;
    cursor: pointer;
}
.filter-row input {
    margin: 0;
    flex: 0 0 auto;
}
.filter-row span:last-child {
    min-width: 0;
}
.swatch {
    display: inline-block;
    width: 11px;
    height: 11px;
    border-radius: 3px;
    flex: 0 0 11px;
}
.count {
    margin-left: auto;
    color: #6B7280;
}
.bank-list {
    max-height: 175px;
    overflow-y: auto;
}
.button-row {
    display: flex;
    gap: 8px;
    margin-top: 10px;
}
.button-row button {
    cursor: pointer;
    padding: 6px 9px;
    border: 1px solid #C8CED6;
    border-radius: 5px;
    background: #F7F8FA;
}
.button-row button:hover {
    background: #E9EDF2;
}
.legend-row {
    display: flex;
    align-items: center;
    gap: 7px;
    margin: 6px 0;
}
</style>
"""

m.get_root().header.add_child(Element(css))
m.get_root().html.add_child(Element(panel_html))

# ==================================================
# 6. JAVASCRIPT FILTERS
# ==================================================

# Generate actual JavaScript layer references, not strings
# that need to be looked up through window[].
layer_mapping_js = "{" + ",".join(
    json.dumps(name) + ":" + layer.get_name()
    for name, layer in bank_layers.items()
) + "}"

js = f"""
(function() {{
    function initialiseBankFilters() {{
        var mapObject = {m.get_name()};
        var bankLayers = {layer_mapping_js};

        var bankChecks = Array.from(
            document.querySelectorAll(".bank-filter")
        );

        var categoryChecks = Array.from(
            document.querySelectorAll(".category-filter")
        );

        function updateBankLayer(bankName, enabled) {{
            var layer = bankLayers[bankName];
            if (!layer) return;

            if (enabled) {{
                if (!mapObject.hasLayer(layer)) {{
                    mapObject.addLayer(layer);
                }}
            }} else {{
                if (mapObject.hasLayer(layer)) {{
                    mapObject.removeLayer(layer);
                }}
            }}
        }}

        function syncCategories() {{
            categoryChecks.forEach(function(categoryCheck) {{
                var category = categoryCheck.dataset.category;

                var matching = bankChecks.filter(function(check) {{
                    return check.dataset.category === category;
                }});

                var enabledCount = matching.filter(function(check) {{
                    return check.checked;
                }}).length;

                categoryCheck.checked =
                    matching.length > 0 &&
                    enabledCount === matching.length;

                categoryCheck.indeterminate =
                    enabledCount > 0 &&
                    enabledCount < matching.length;
            }});
        }}

        bankChecks.forEach(function(check) {{
            check.addEventListener("change", function() {{
                updateBankLayer(check.dataset.bank, check.checked);
                syncCategories();
            }});
        }});

        categoryChecks.forEach(function(check) {{
            check.addEventListener("change", function() {{
                var category = check.dataset.category;

                bankChecks.forEach(function(bankCheck) {{
                    if (bankCheck.dataset.category === category) {{
                        bankCheck.checked = check.checked;
                        updateBankLayer(
                            bankCheck.dataset.bank,
                            check.checked
                        );
                    }}
                }});

                syncCategories();
            }});
        }});

        document.getElementById("select-all")
            .addEventListener("click", function() {{
                bankChecks.forEach(function(check) {{
                    check.checked = true;
                    updateBankLayer(check.dataset.bank, true);
                }});
                syncCategories();
            }});

        document.getElementById("clear-all")
            .addEventListener("click", function() {{
                bankChecks.forEach(function(check) {{
                    check.checked = false;
                    updateBankLayer(check.dataset.bank, false);
                }});
                syncCategories();
            }});

        syncCategories();
    }}

    // Run after the page's map and controls are available.
    if (document.readyState === "loading") {{
        document.addEventListener(
            "DOMContentLoaded",
            initialiseBankFilters
        );
    }} else {{
        initialiseBankFilters();
    }}
}})();
"""

# Add as executable JavaScript to the Folium script section.
m.get_root().script.add_child(Element(js))

# ==================================================
# 7. SAVE THE HTML FILE
# ==================================================

output_file = "european_banking_map.html"
m.save(output_file)

print(f"Map saved successfully: {output_file}")


Map saved successfully: european_banking_map2.html


In [9]:
print('ok')

ok
